# Classical subtropical Stommel gyre

This example computes the planetary-geostrophic analogue of a classical Stommel subtropical gyre. A flat rectangular basin on a beta plane is forced by a zonal wind stress with negative curl throughout the interior. Buoyancy forcing is zero, so the circulation is barotropic and the solution isolates the Sverdrup interior and frictional western boundary current.

With the convention $(U,V)=(-\partial_y\Psi,\partial_x\Psi)$, the inviscid interior satisfies

$$\beta V = \hat{\boldsymbol z}\mathbin{\boldsymbol\cdot}(\boldsymbol\nabla\mathbin{\boldsymbol\times}\boldsymbol\tau),$$

while the analytical PG bottom stress closes the circulation near the basin boundary.

In [ ]:
using Pkg
Pkg.activate(joinpath(@__DIR__, "..", "PlanetaryGeostrophy"))
using PlanetaryGeostrophy
using Plots

## Basin and forcing

The wind changes from westward at the southern boundary to eastward at the northern boundary. Its curl is negative between the boundaries, producing a single anticyclonic subtropical gyre.

In [ ]:
Lx, Ly = 2.0, 1.0
H0 = 1.0
f0, beta = 1.0, 0.30
Ek, Γ = 0.01, 0.20
tau0 = 5e-4

eta(y) = (y + Ly / 2) / Ly
tau_x(x, y) = -tau0 * cos(pi * eta(y))
tau_y(x, y) = 0.0
wind_curl(x, y) = -tau0 * pi / Ly * sin(pi * eta(y))

problem = PGProblem(
    parameters=PGParameters(Ek, Γ; viscosity=1.0, diffusivity=1.0),
    depth=(x, y) -> H0,
    depth_x=(x, y) -> 0.0,
    depth_y=(x, y) -> 0.0,
    coriolis=(x, y) -> f0 + beta * y,
    coriolis_x=(x, y) -> 0.0,
    coriolis_y=(x, y) -> beta,
    wind_stress=(x, y) -> tau_x(x, y) + im * tau_y(x, y),
    restoring_buoyancy=(x, y, z) -> 0.0,
    surface_buoyancy=(x, y) -> 0.0,
    boundary=rectangle_boundary(Lx, Ly),
)

## Surface-pressure and transport solution

In [ ]:
solution = solve_pg(
    problem;
    mesh_file=joinpath(@__DIR__, "stommel_pg_pressure.msh"),
    mesh_resolution=0.06,
    pressure_order=2,
    overwrite_mesh=true,
)
display(performance_summary(solution))

In [ ]:
# Stay just inside the finite-element boundary for point evaluation.
x = range(-0.99Lx / 2, 0.99Lx / 2; length=49)
y = range(-0.99Ly / 2, 0.99Ly / 2; length=33)
diagnostics = compute_diagnostics(solution, x, y)
display(performance_summary(diagnostics))

## PG diagnostics

The transport vectors and streamfunction show the broad Sverdrup interior and narrow western return current. Bottom stress supplies the vorticity needed to close the gyre.

In [ ]:
bottom_stress_figure = plot_bottom_stress(
    diagnostics;
    arrow_scale=0.35,
    output=joinpath(@__DIR__, "stommel_pg_bottom_stress.png"),
)
transport_figure = plot_depth_integrated_current(
    diagnostics;
    arrow_scale=0.35,
    output=joinpath(@__DIR__, "stommel_pg_transport.png"),
)
streamfunction_figure = plot_streamfunction(
    diagnostics;
    output=joinpath(@__DIR__, "stommel_pg_streamfunction.png"),
)
diagnostic_figure = plot_diagnostics(
    diagnostics;
    arrow_scale=0.35,
    output=joinpath(@__DIR__, "stommel_pg_diagnostics.png"),
)
display(diagnostic_figure)

## Sverdrup-interior comparison

Away from the western boundary, $V_{Sv}=\operatorname{curl}(\tau)/\beta$. Integrating $\partial_x\Psi_{Sv}=V_{Sv}$ from the eastern boundary gives an interior streamfunction that does not satisfy the western no-normal-flow condition. The difference from the PG solution therefore highlights the frictional western boundary layer.

In [ ]:
surface_pressure_values = [surface_pressure(solution, xx, yy) for xx in x, yy in y]
sverdrup_V = [wind_curl(xx, yy) / beta for xx in x, yy in y]
sverdrup_streamfunction = [
    (xx - Lx / 2) * wind_curl(xx, yy) / beta for xx in x, yy in y
]
transport_speed = hypot.(diagnostics.transport_U, diagnostics.transport_V)

p1 = heatmap(x, y, surface_pressure_values';
    title="surface pressure", xlabel="x", ylabel="y", aspect_ratio=:equal)
p2 = contourf(x, y, diagnostics.streamfunction';
    title="PG streamfunction", xlabel="x", ylabel="y", aspect_ratio=:equal)
p3 = contourf(x, y, sverdrup_streamfunction';
    title="Sverdrup interior", xlabel="x", ylabel="y", aspect_ratio=:equal)
p4 = heatmap(x, y, transport_speed';
    title="transport speed", xlabel="x", ylabel="y", aspect_ratio=:equal)
comparison_figure = plot(p1, p2, p3, p4; layout=(2, 2), size=(1000, 800))
savefig(comparison_figure, joinpath(@__DIR__, "stommel_pg_sverdrup_comparison.png"))
display(comparison_figure)

In [ ]:
western = x .< -Lx / 4
eastern = x .> Lx / 4
western_peak = maximum(transport_speed[western, :])
eastern_peak = maximum(transport_speed[eastern, :])
western_intensification = western_peak / eastern_peak
display((; western_peak, eastern_peak, western_intensification))